In [1]:
import re
import argparse
import os
import datetime

# backend 0 default, backend 1 pp, backend 2, 5, 6 DP, backend 3 TP

def custom_print(message, file_path=None, mode='a'):
    print(message)  # Print to terminal
    if file_path:
        with open(file_path, mode) as file:
            file.write(message + '\n')

def backend_to_comm_type(backend, rank_tid=None):
    if backend == 0:
        return "Default"
    elif backend % 10 == 1:
        return "PP"
    # elif backend in [2, 5, 6]:
    #     return "DP"
    elif backend == 2:
        return "TP"
    else:
        return "unknown"

def time_to_ms(t):
    return (
        t.hour * 3600_000 +
        t.minute * 60_000 +
        t.second * 1000 +
        t.microsecond / 1000.0
    )
    
def calculate_pp_backend(backend, pp_comm_group = -1):
    if pp_comm_group == -1:
        return backend
    else:
        return pp_comm_group * 10 + backend
    
def is_same_comm_type(backend1, backend2):
    if backend1 is None or backend2 is None:
        return False
    type1 = backend_to_comm_type(backend1)
    type2 = backend_to_comm_type(backend2)

    if type1 == type2 and type1 == "PP":
        return backend1 == backend2
    
    return type1 == type2


def parse_communication_pattern(log_file, rank_tid):
    pattern = re.compile(r"\x1b\[[0-9;]*m?Backend (\d+) RANK (\d+) tid: (\d+).*?Idx: (\d+), issuing")
    # pattern = re.compile(r"\x1b\[[0-9;]*m?Backend (\d+) RANK (\d+) tid: (\d+).*? pre_coll, Idx: (\d+)")
    communication_pattern = []

    time_pat = re.compile(
        r"t:\[(\d+:\d+:\d+\.\d+)\]"
    )

    with open(log_file, 'r') as file:
        start_backend = None
        idx_list = None
        end_backend = None

        start_time = None
        end_time = None

        # Track per-idx timestamps
        idx_start_times = {}
        idx_end_times = {}

        time_gap = {}

        for line in file:
            match = pattern.search(line)
            if match:
                backend = int(match.group(1))
                tid = int(match.group(3))
                idx = int(match.group(4))

                # Check if the line contains "PP comm group"
                pp_comm_group_match = re.search(r"PP comm group: (\d+)", line)
                if pp_comm_group_match:
                    pp_comm_group = int(pp_comm_group_match.group(1))
                else:
                    pp_comm_group = -1

                # remove TP traffic and backend 0 traffic, PP allreduce traffic
                if backend_to_comm_type(backend) == "TP" or backend == 0:
                    continue
                if backend_to_comm_type(backend) == "PP" and "allreduce" in line:
                    continue

                if (backend == 1):
                    assert(pp_comm_group != -1)
                    backend = calculate_pp_backend(backend, pp_comm_group) # has information of pp group

                # extract timestamp if present
                tmatch = time_pat.search(line)
                ts = None
                if tmatch:
                    ts = datetime.datetime.strptime(tmatch.group(1), "%H:%M:%S.%f")
                    ts = time_to_ms(ts)

                # record pre/post timestamps
                # if "pre_coll" in line and ts:
                
                #     idx_start_times[(backend, idx)] = ts
                # elif "post_coll" in line and ts:
                #     idx_end_times[(backend, idx)] = ts

                if tid == rank_tid:
                    if not is_same_comm_type(backend, start_backend):
                        # transition to a new backend
                        if start_backend is not None:
                            communication_pattern.append((start_backend, idx_list, end_backend))
                        start_backend = backend
                        idx_list = [idx]
                        end_backend = backend
                    else:
                        idx_list.append(idx)
                        end_backend = backend

        # Append the last backend, rank, and idx
        if start_backend is not None:
            communication_pattern.append((start_backend, idx_list, end_backend))

    return communication_pattern

def get_rank_tid(log_file, node_id, num_local_ranks):
    pattern = re.compile(r"Backend 0 RANK (\d+) tid: (\d+)")
    with open(log_file, 'r') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                backend = int(match.group(1))
                tid = int(match.group(2))
                return [tid] * num_local_ranks  # Assuming all local ranks share the same pattern
    raise ValueError(f"No matching line found in log file {log_file}")

# List all files in the folder that match the pattern torchrun_nid*
log_folder = "<path-to-opus>/Opus/torchtitan/opus-test/dp-4-pp-8-tp-4-pm/output-opus-l_0-no-ctl/"
torchrun_files = [f for f in os.listdir(log_folder) if f.startswith("torchrun_nid")]
torchrun_files.sort()

print(torchrun_files)
num_local_ranks = 4

output_folder = "comm_pattern"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

for i, log_file in enumerate(torchrun_files):
    log_file_full_path = os.path.join(log_folder, log_file)
    rank_tid_list = get_rank_tid(log_file_full_path, i, num_local_ranks)
    print(rank_tid_list)

    for rank, tid in enumerate(rank_tid_list):
        # rank is local rank
        output_file = f"{output_folder}/comm_pattern_{i}_rank{rank}.txt"
        # Clean the output file before writing
        with open(output_file, 'w') as file:
            file.truncate(0)
        custom_print(f"Node: {i}, Rank: {rank}, Tid: {tid}", output_file)
        pattern = parse_communication_pattern(log_file_full_path, tid)

        for backend, idx_list, end_backend in pattern:
            custom_print(f"{backend_to_comm_type(backend)} start_backend: {backend} start_idx: {idx_list[0]} end_backend: {end_backend} end_idx: {idx_list[-1]} len: {len(idx_list)}", output_file)


['torchrun_nid001356.ans', 'torchrun_nid001360.ans', 'torchrun_nid001580.ans', 'torchrun_nid001584.ans', 'torchrun_nid001588.ans', 'torchrun_nid001605.ans', 'torchrun_nid001628.ans', 'torchrun_nid001629.ans', 'torchrun_nid001748.ans', 'torchrun_nid001749.ans', 'torchrun_nid001880.ans', 'torchrun_nid001881.ans', 'torchrun_nid002097.ans', 'torchrun_nid002100.ans', 'torchrun_nid002113.ans', 'torchrun_nid002116.ans', 'torchrun_nid002145.ans', 'torchrun_nid002217.ans', 'torchrun_nid002220.ans', 'torchrun_nid002224.ans', 'torchrun_nid002225.ans', 'torchrun_nid002397.ans', 'torchrun_nid002408.ans', 'torchrun_nid002409.ans', 'torchrun_nid002412.ans', 'torchrun_nid002525.ans', 'torchrun_nid002528.ans', 'torchrun_nid002529.ans', 'torchrun_nid003476.ans', 'torchrun_nid003477.ans', 'torchrun_nid003848.ans', 'torchrun_nid003849.ans']
[140382519318336, 140382519318336, 140382519318336, 140382519318336]
Node: 0, Rank: 0, Tid: 140382519318336
PP start_backend: 11 start_idx: 1000000 end_backend: 11 end

In [ ]:
import re
import argparse
import datetime
import numpy as np

# backend 0 default, backend 1 pp, backend 2, 5, 6 DP, backend 3 TP

def custom_print(message, file_path=None, mode='a'):
    print(message)  # Print to terminal
    if file_path:
        with open(file_path, mode) as file:
            file.write(message + '\n')

def backend_to_comm_type(backend, rank_tid=None):
    if backend == 0:
        return "Default"
    elif backend == 1:
        return "PP"
    elif backend in [2, 5, 6]:
        return "DP"
    elif backend == 3:
        return "TP"
    else:
        return "unknown"
    
def is_same_comm_type(backend1, backend2):
    return backend_to_comm_type(backend1) == backend_to_comm_type(backend2)

def time_to_ms(t):
    return (
        t.hour * 3600_000 +
        t.minute * 60_000 +
        t.second * 1000 +
        t.microsecond / 1000.0
    )

def parse_communication_pattern(log_file, rank_tid):
    main_pat = re.compile(
        r"Backend (\d+) RANK (\d+) tid: (\d+) .*?Idx: (\d+),"
    )
    time_pat = re.compile(
        r"t:\[(\d+:\d+:\d+\.\d+)\]"
    )

    communication_pattern = []

    start_backend = None
    end_backend = None

    idx_list = None

    start_time = None
    end_time = None

    # Track per-idx timestamps
    idx_start_times = {}
    idx_end_times = {}

    time_gap = {}

    with open(log_file, 'r') as file:
        for line in file:
            match = main_pat.search(line)
            if not match:
                continue

            backend = int(match.group(1))
            tid = int(match.group(3))
            idx = int(match.group(4))

            # filter
            if backend_to_comm_type(backend) == "TP" or backend == 0:
                continue
            if tid != rank_tid:
                continue

            # extract timestamp if present
            tmatch = time_pat.search(line)
            ts = None
            if tmatch:
                ts = datetime.datetime.strptime(tmatch.group(1), "%H:%M:%S.%f")
                ts = time_to_ms(ts)

            # record pre/post timestamps
            if "pre_coll" in line and ts:
                idx_start_times[(backend, idx)] = ts
            elif "post_coll" in line and ts:
                idx_end_times[(backend, idx)] = ts

            # comm-type transition
            if "pre_coll" in line:
                if not is_same_comm_type(backend, start_backend):
                    if start_backend is not None:
                        communication_pattern.append(
                            (start_backend, idx_list, end_backend, start_time, end_time)
                        )
                    
                    start_time = ts
                    if end_time:
                        time_gap[(start_backend, idx_list[0])] = start_time - end_time
                    start_backend = backend
                    idx_list = [idx]
                else:
                    end_backend = backend
                    idx_list.append(idx)

            if "post_coll" in line:
                end_time = ts

        # flush last segment
        if start_backend is not None:
            communication_pattern.append(
                (start_backend, idx_list, end_backend, start_time, end_time)
            )

    return communication_pattern, time_gap

# log_file = f"<path-to-opus>/Opus/torchtitan/opus-test/dp-2-pp-2-tp-4-pm-8b-provision/output-opus-v3-l_0-b24-provision/torchrun_nid001012.ans"

output_file = "comm_pattern_pp_0.txt"
with open(output_file, 'w') as file:
    file.truncate(0)
log_file = f"<path-to-opus>/Opus/torchtitan/opus-test/deepseek-dp-2-pp-2-tp-4-pm/output-no-controller/torchrun_nid001028.ans"

pattern, time_gap_map_pp0 = parse_communication_pattern(log_file, 140114140198464)

for backend, idx_list, end_backend, start_time, end_time in pattern:
    custom_print(f"{backend_to_comm_type(backend)} start_backend: {backend} start_idx: {idx_list[0]} end_backend: {end_backend} end_idx: {idx_list[-1]} len: {len(idx_list)}, start_time: {start_time}, end_time: {end_time}", output_file)

# if time_gap_list:
#     avg_gap = np.mean(time_gap_list)
#     std_gap = np.std(time_gap_list)
#     min_gap = np.min(time_gap_list)
#     max_gap = np.max(time_gap_list)

#     custom_print(f"Average time gap: {avg_gap:.3f} ms", output_file)
#     custom_print(f"Standard deviation: {std_gap:.3f} ms", output_file)
#     custom_print(f"Minimum time gap: {min_gap:.3f} ms", output_file)
#     custom_print(f"Maximum time gap: {max_gap:.3f} ms", output_file)
# else:
#     custom_print("No time gaps to calculate statistics.")


output_file = "comm_pattern_pp_1.txt"
with open(output_file, 'w') as file:
    file.truncate(0)
log_file = f"<path-to-opus>/Opus/torchtitan/opus-test/deepseek-dp-2-pp-2-tp-4-pm/output-no-controller/torchrun_nid001032.ans"

pattern, time_gap_map_pp1 = parse_communication_pattern(log_file, 140583809054272)

for backend, idx_list, end_backend, start_time, end_time in pattern:
    custom_print(f"{backend_to_comm_type(backend)} start_backend: {backend} start_idx: {idx_list[0]} end_backend: {end_backend} end_idx: {idx_list[-1]} len: {len(idx_list)}, start_time: {start_time}, end_time: {end_time}", output_file)

common_gap = []
for (key1, gap1)in time_gap_map_pp0.items():
    for (key2, gap2) in time_gap_map_pp1.items():
        if key1 == key2 and key1[0] == 1:  # Check if backend and idx are the same
            min_gap = min(gap1, gap2)
            custom_print(f"Backend: {key1[0]}, Idx: {key1[1]}, Min Gap: {min_gap:.3f} ms", output_file)
            common_gap.append(min_gap)

if common_gap:
    avg_gap = np.mean(common_gap)
    std_gap = np.std(common_gap)
    min_gap = np.min(common_gap)
    max_gap = np.max(common_gap)

    custom_print(f"Average time gap: {avg_gap:.3f} ms", output_file)
    custom_print(f"Standard deviation: {std_gap:.3f} ms", output_file)
    custom_print(f"Minimum time gap: {min_gap:.3f} ms", output_file)
    custom_print(f"Maximum time gap: {max_gap:.3f} ms", output_file)